# Evaluator walkthrough: RAD-DINO on VinDr-CXR

This notebook connects all the parts of `radharmony.evaluator` in one example. It runs a **linear probe** on
the frozen features of [`microsoft/rad-dino`](https://huggingface.co/microsoft/rad-dino) on VinDr-CXR. It
builds the transform and the encoder by hand, so that you can see each part.

| Part | What it does |
|---|---|
| `EncoderPreprocessTransform.from_huggingface` | Puts the Hugging Face image processor of the model in a MONAI step. The step reads the DICOM file, resizes it and normalizes it as RAD-DINO needs |
| `RadiologyEncoderTransform` | Puts this step in a transform for the dataset, with the RadHarmony augmentation methods and the output keys |
| `ImageEncoderWrapper.from_huggingface` | Wraps the Hugging Face model in an `nn.Module` that returns one feature for each image, `Tensor[B, D]` |
| `LinearProbeEvaluator` | Fits one logistic regression for each label on the frozen features, in k-fold mode or in fixed-split mode |

A backbone recipe gives the transform and the encoder in one call, for example `make_raddino()`. See
[Backbones](https://f10409.github.io/RadHarmony/reference/backbones/index.html). To wrap a model that has no
recipe, see [Add a backbone](https://f10409.github.io/RadHarmony/how-to/add-a-backbone.html).

**Requirements**

- RadHarmony with the `notebook` and `model` extras
  ([Install](https://f10409.github.io/RadHarmony/get-started/install.html)).
- VinDr-CXR on your disk
  ([Get the data: VinDr-CXR](https://f10409.github.io/RadHarmony/how-to/get-the-data.html#vindr-cxr)). It
  needs a PhysioNet account with the CITI training and the signed data use agreement.
- A GPU. The first run downloads the RAD-DINO weights from Hugging Face.

## 1. Settings

Set the 2 VinDr-CXR folders to the paths on your computer. The other settings keep the run short. With these
values, the notebook uses 2,000 train images and 500 test images, and 7 of the 28 labels. Set
`N_TRAIN_PATIENTS`, `N_TEST_PATIENTS` and `LABEL_SUBSET` to `None` to use all the data.

In [ ]:
VINDR_TRAIN_DIR = "/path/to/vindr-cxr/1.0.0/train/"
VINDR_TEST_DIR = "/path/to/vindr-cxr/1.0.0/test/"
EMBEDDING_CACHE = "./cache/vindr_raddino"  # the feature files go here

# Settings for a short run. Use larger values for a real benchmark.
N_TRAIN_PATIENTS = 2000  # None uses all 15,000 train images
N_TEST_PATIENTS = 500  # None uses all 3,000 test images
LABEL_SUBSET = [
    "aortic_enlargement",
    "cardiomegaly",
    "lung_opacity",
    "pleural_effusion",
    "pleural_thickening",
    "pulmonary_fibrosis",
    "tuberculosis",
]
N_FOLDS = 5
N_TRAIN_SAMPLES = [500, 1500]  # one run for each train size, in images
BATCH_SIZE = 32
DEVICE = "cuda"

## 2. Build the transform

`EncoderPreprocessTransform.from_huggingface` reads, resizes and normalizes each image with the image
processor of RAD-DINO. VinDr-CXR has DICOM files, so use `reader="monai"`. The default reader, `"pil"`, cannot
open DICOM.

`RadiologyEncoderTransform` puts this step in a transform for the dataset. Its `with_*` methods add
augmentation steps, for example `.with_flip()`. Use them only to fine-tune.

A frozen-feature probe gets the feature of each image one time. So a random step runs only one time for each
image, and it does not help. If the transform has a random step, the evaluator warns
([D-41](https://f10409.github.io/RadHarmony/design.html#d-41-the-embedding-cache-holds-features-only)). See
[Evaluators: Embedding cache](https://f10409.github.io/RadHarmony/reference/evaluators/index.html#embedding-cache).

In [ ]:
from transformers import AutoImageProcessor

from radharmony.evaluator import EncoderPreprocessTransform, RadiologyEncoderTransform

# To change the image size, give size=... to from_pretrained.
processor = AutoImageProcessor.from_pretrained("microsoft/rad-dino")
pp = EncoderPreprocessTransform.from_huggingface(keys=["img"], processor=processor, reader="monai")

# No with_* augmentation: this transform is for a frozen-feature probe.
transform = RadiologyEncoderTransform(preprocess=pp, output_keys={"img", "cls"}).get_transform()

## 3. Load the dataset

Give the transform to `VinDrCXRTrainDataset`. Set `output_cls=True`, because the evaluator reads the labels
from the `cls` key. `cache_dir=None` turns off the disk cache. The embedding cache in section 5 keeps the
features.

`keep_patients()` returns the rows of some random patients. `set_harmonized_df()` gives this smaller
harmonized table to the dataset. The tour notebook uses the same helper. VinDr-CXR has no patient ID, so each
image counts as one patient. See
[Save and reuse a harmonized table](https://f10409.github.io/RadHarmony/how-to/save-and-reuse.html) for other
ways to filter the table.

In [ ]:
from radharmony.dataset import VinDrCXRTrainDataset


def keep_patients(df, n, seed=0):
    """Return the rows of n random patients of a harmonized table. n=None returns all rows."""
    if n is None:
        return df
    ids = df["patient_id"].drop_duplicates().sample(n=n, random_state=seed)
    return df[df["patient_id"].isin(ids)].reset_index(drop=True)


train_ds = VinDrCXRTrainDataset(
    base_image_dir=VINDR_TRAIN_DIR,
    transform=transform,
    cache_dir=None,
    output_cls=True,
)
train_ds.set_harmonized_df(keep_patients(train_ds.get_harmonized_df(), N_TRAIN_PATIENTS))
df = train_ds.get_harmonized_df()
print(f"train_ds: {len(df)} images, {len(train_ds.LABEL_COLS)} labels")
print(df[LABEL_SUBSET].sum().astype(int).to_dict())  # positive images for each label

## 4. Wrap the encoder

`ImageEncoderWrapper.from_huggingface` calls `model(pixel_values=x)`. `pool="cls"` takes the CLS token of
`last_hidden_state` as the feature. The wrapper is an `nn.Module`, so `.to(device)` and `.parameters()` act on
the model inside it. `FinetuneEvaluator` needs this. A checkpoint save also needs it. See
[Evaluators: Encoder contract](https://f10409.github.io/RadHarmony/reference/evaluators/index.html#encoder-contract).

In [ ]:
from transformers import AutoModel

from radharmony.evaluator import ImageEncoderWrapper

encoder = ImageEncoderWrapper.from_huggingface(
    AutoModel.from_pretrained("microsoft/rad-dino"),
    pool="cls",
).to(DEVICE)
print(f"encoder: {type(encoder.model).__name__}")

## 5. Linear probe in k-fold mode

`dataset=` sets k-fold mode. The evaluator gets the features one time and writes them to
`EMBEDDING_CACHE.pkl`. Then it splits the patients into `n_folds` folds. In each fold, the probe takes one
random train subset for each size in `n_train_samples`, and fits on it. The probe keeps 10% of the train
patients apart to pick the threshold (Youden's J by default) of F1
([D-29](https://f10409.github.io/RadHarmony/design.html#d-29-the-threshold-comes-from-validation-data)).

A second run with the same `EMBEDDING_CACHE` reads the features from the file. This also works when you change
the labels. See [Linear probe](https://f10409.github.io/RadHarmony/reference/evaluators/linear_probe.html) for
all the arguments.

In [ ]:
from radharmony.evaluator import LinearProbeEvaluator

probe = LinearProbeEvaluator(
    image_encoder=encoder,
    dataset=train_ds,
    labels=LABEL_SUBSET,
    n_folds=N_FOLDS,
    n_train_samples=N_TRAIN_SAMPLES,
    batch_size=BATCH_SIZE,
    num_workers=8,  # more workers read the DICOM files faster
    device=DEVICE,
    embedding_cache=EMBEDDING_CACHE,
)
df = probe.evaluate()
df.head()

In [ ]:
# One row for each train size, fold and label, and a macro_average row for each run.
print(df.shape)
df.groupby(["n_train", "label"])["auroc"].mean().unstack(0).round(3)

## 6. Summary for each label

`save_results` writes 2 files: `results.csv` (the result table) and `results_summary.csv`. The summary has one
row for each label and train size. It gives the mean, the std and the 95% CI of each metric over the folds.
See
[Evaluators: Result table](https://f10409.github.io/RadHarmony/reference/evaluators/index.html#result-table).

The validation part is the 10% of the train patients that pick the threshold. With a small train size, it can
have fewer than 2 positives or fewer than 2 negatives of a label. Then F1 and the other threshold metrics are
NaN, and `save_results()` warns. AUROC and AUPRC use no threshold, so they have a value
([Threshold](https://f10409.github.io/RadHarmony/reference/evaluators/index.html#threshold)).

In [ ]:
import pandas as pd

out_dir = probe.save_results(df, output_dir="./eval_out/raddino_kfold")
summary = pd.read_csv(f"{out_dir}/results_summary.csv")
cols = ["label", "n_train", "auroc_mean", "auroc_std", "auprc_mean", "auprc_std", "f1_mean", "f1_std"]
summary[cols].round(3)

## 7. Fixed-split mode (optional)

Give `train_dataset=` and `test_dataset=` in place of `dataset=` to use fixed-split mode. The official
VinDr-CXR test split becomes the test set. The spread of the scores comes from 2 sources: `n_seeds` (other
train subsets) and `n_bootstrap` (samples of the test patients, with replacement). The result table has the
same columns, so `save_results` works in the same way. See
[Evaluators: Split modes](https://f10409.github.io/RadHarmony/reference/evaluators/index.html#split-modes).

The features go to `EMBEDDING_CACHE.train.pkl` and `EMBEDDING_CACHE.test.pkl`.

This cell can print some lines of `Exception ignored in _MultiProcessingDataLoaderIter.__del__` with
`AssertionError: can only test a child process`. They come from the data loader of the first probe in the same
kernel. They do no harm, and the results are right.

In [ ]:
from radharmony.dataset import VinDrCXRTestDataset

test_ds = VinDrCXRTestDataset(
    base_image_dir=VINDR_TEST_DIR,
    transform=transform,
    cache_dir=None,
    output_cls=True,
)
test_ds.set_harmonized_df(keep_patients(test_ds.get_harmonized_df(), N_TEST_PATIENTS))

probe_fixed = LinearProbeEvaluator(
    image_encoder=encoder,
    train_dataset=train_ds,
    test_dataset=test_ds,
    labels=LABEL_SUBSET,
    n_train_samples=[1500],
    n_seeds=3,
    n_bootstrap=100,
    batch_size=BATCH_SIZE,
    num_workers=8,
    device=DEVICE,
    embedding_cache=EMBEDDING_CACHE,
)
df_fixed = probe_fixed.evaluate()
out_dir = probe_fixed.save_results(df_fixed, output_dir="./eval_out/raddino_fixed")
summary = pd.read_csv(f"{out_dir}/results_summary.csv")
summary[["label", "auroc_mean", "auroc_ci_lo", "auroc_ci_hi", "f1_mean"]].round(3)

## Next steps

- [Run a probe](https://f10409.github.io/RadHarmony/how-to/run-a-probe.html): the kNN, SVM and prototype
  probes, and zero-shot, on the same features.
- [Compare backbones](https://f10409.github.io/RadHarmony/how-to/compare-backbones.html): run more encoders on
  the same data.
- [Run segmentation](https://f10409.github.io/RadHarmony/how-to/run-segmentation.html): the segmentation
  probes on patch features.